# Data Quality — Tokopedia
Jalankan semua sel dengan kernel env.
Output final mengikuti kontrak canonical 12 kolom; `product_id` memakai SKU master.


In [1]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pipeline/validation/analysis.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.validation.analysis import (
    load_analysis, standard_data, missing_values, quality_issues,
    type_report, summary, export_analysis,
)

SOURCE = "tokopedia"
hasil = load_analysis(SOURCE, ROOT / "data/source")
data_bersih = standard_data(hasil)


## 1. Missing value

Field wajib: order ID, tanggal, produk, quantity, harga satuan, status; total transaksi Website juga wajib. Semua field master wajib. Baris yang tidak memenuhi syarat ditolak. Customer/kota/email yang tidak tersedia tetap kosong, tanpa dummy. Field opsional kosong yang tersedia dalam source dicatat sebagai warning.

In [2]:
display(missing_values(hasil))
display(quality_issues(hasil, "missing"))

,sumber,kolom,jumlah_kosong
0,tokopedia,transaction_id,0
1,tokopedia,transaction_date,0
2,tokopedia,item_name,0
3,tokopedia,quantity,0
4,tokopedia,price,0
5,tokopedia,buyer_name,1
6,tokopedia,city,1
7,tokopedia,payment,1
8,tokopedia,status,0


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,12,WARNING,buyer_name,MISSING_OPTIONAL,
1,tokopedia,16,WARNING,city,MISSING_OPTIONAL,
2,tokopedia,20,WARNING,payment,MISSING_OPTIONAL,


## 2. Duplicate

Business key: `(channel, order_id)` untuk dataset satu item per order saat ini; master memakai `product_id`/SKU. Record yang sama disimpan satu kali. Jika key sama tetapi nilainya berbeda, semua versi ditolak untuk ditinjau. Bila kelak order memiliki beberapa item, tambahkan line ID dari sumber.

In [3]:
display(quality_issues(hasil, "duplicate"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,51,INFO,business_key,DUPLICATE_BUSINESS_KEY,TKP-0106


## 3. Invalid value

Quantity wajib integer positif. Harga wajib positif dan maksimal dua desimal. Total harus sama dengan quantity × harga satuan. Status dataset saat ini: `Completed`, `Cancelled`, `Returned`; status lain ditolak. Harga berbeda dari master diberi warning karena mungkin promo. Total harga adalah nilai bruto; hitung penjualan selesai hanya dari `Completed`.

In [4]:
display(quality_issues(hasil, "invalid"))

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,4,ERROR,quantity,NON_POSITIVE,0
1,tokopedia,8,ERROR,price,NON_POSITIVE,-50000
2,tokopedia,24,ERROR,status,INVALID_STATUS,in_progress


## 4. Date format

Tanggal Shopee/Tokopedia: DD/MM/YYYY; Website: MMM DD, YYYY; Offline: DD-MMM-YYYY. Hasil CSV selalu YYYY-MM-DD. Tanggal tidak valid ditolak. Product Master tidak memiliki tanggal transaksi.

In [5]:
display(quality_issues(hasil, "date"))
if "tanggal_order" in data_bersih:
    display(data_bersih[["order_id", "tanggal_order"]].head(5))
else:
    print("Tidak berlaku: master produk tidak memiliki tanggal transaksi.")

,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,32,ERROR,transaction_date,INVALID_DATE,not-a-date


,order_id,tanggal_order
0,TKP-0101,2026-02-19
1,TKP-0102,2026-01-26
2,TKP-0103,2026-06-02
3,TKP-0105,2026-02-20
4,TKP-0106,2026-01-17


## 5. Data type

Identifier string, quantity Int64, tanggal datetime, dan uang Decimal. CSV tidak menyimpan tipe data; ekspor tanggal menggunakan YYYY-MM-DD. Teks dirapikan spasinya tanpa merusak nama brand, SKU, shade, atau SPF/PA++++.

In [6]:
display(type_report(data_bersih))

,kolom,dtype,tipe_nilai
0,order_id,string,str
1,product_id,string,str
2,product_name,string,str
3,kategori,string,str
4,quantity,Int64,int64
5,total_harga,object,Decimal
6,tanggal_order,datetime64[us],Timestamp
7,kota,string,str
8,channel,string,str
9,status,string,str


## 6. Product consistency

Variasi huruf besar/kecil, spasi, underscore, dan hyphen dicocokkan ke master. Nama produk dan kategori mengikuti master. Produk tidak dikenal/typo ambigu ditolak, tanpa menebak SKU. Pada master, SKU harus unik.

In [7]:
display(data_bersih[["product_id", "product_name", "kategori"]].drop_duplicates())
display(quality_issues(hasil, "product"))

,product_id,product_name,kategori
0,CRY-SKC-001,Crystallure Supreme Revitalizing Oil Serum 20ml,Skincare
1,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup
2,INS-MUP-001,Instaperfect Skincover Air Cushion 02 Beige,Makeup
3,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup
4,BDF-BDY-001,Biodef Body Wash Fresh Care 450ml,Body Care
5,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance
6,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup
7,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup
10,EMN-SKC-001,Emina Bright Stuff Face Wash 100ml,Skincare
12,EMN-MUP-002,Emina Glossy Stain 01 Autumn Bell,Makeup


,sumber,baris,tingkat,kolom,masalah,nilai_asli
0,tokopedia,28,ERROR,item_name,UNMAPPED_PRODUCT,Brigthning Serumm 30ml


## Hasil akhir

Format dan urutan kolom sama untuk semua transaksi. `kota` adalah kota pelanggan online atau kota toko offline; Website yang tidak memiliki kota tetap kosong. Nama channel tetap Shopee/Tokopedia/Website/Offline Store agar bisa dibandingkan.

Hasil utama: `data/processed/clean/`. Notebook ini menyimpan file bersih sumber yang dibahas; `analisa.ipynb` menyimpan seluruh sumber, `sales.csv`, `summary.csv`, serta satu `quality_issues.csv` untuk detail masalah.

In [8]:
ringkasan = summary(hasil)
assert (ringkasan["awal"] == ringkasan["bersih"] + ringkasan["duplikat"] + ringkasan["ditolak"]).all()
display(ringkasan)
display(data_bersih.head(10))
folder_hasil = export_analysis(hasil)
print("Tersimpan:", folder_hasil)

,sumber,awal,bersih,duplikat,ditolak
0,tokopedia,51,45,1,5


,order_id,product_id,product_name,kategori,quantity,total_harga,tanggal_order,kota,channel,status,customer_email,harga_satuan
0,TKP-0101,CRY-SKC-001,Crystallure Supreme Revitalizing Oil Serum 20ml,Skincare,3,657000.00,2026-02-19,Surabaya,Tokopedia,Completed,<NA>,219000.00
1,TKP-0102,EMN-MUP-001,Emina Cheek Lit Cream Blush Peach,Makeup,1,46900.00,2026-01-26,Surabaya,Tokopedia,Completed,<NA>,46900.00
2,TKP-0103,INS-MUP-001,Instaperfect Skincover Air Cushion 02 Beige,Makeup,1,149000.00,2026-06-02,Jakarta,Tokopedia,Cancelled,<NA>,149000.00
3,TKP-0105,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup,3,507000.00,2026-02-20,Bandung,Tokopedia,Returned,<NA>,169000.00
4,TKP-0106,BDF-BDY-001,Biodef Body Wash Fresh Care 450ml,Body Care,3,107700.00,2026-01-17,Semarang,Tokopedia,Completed,<NA>,35900.00
5,TKP-0107,WND-BDY-001,Wonderly Body Mist Sweet Blossom 100ml,Fragrance,1,45900.00,2026-02-20,Jakarta,Tokopedia,Completed,<NA>,45900.00
6,TKP-0109,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup,2,125800.00,2026-04-21,Semarang,Tokopedia,Completed,<NA>,62900.00
7,TKP-0110,WRD-MUP-002,Wardah Colorfit Perfect Glow Cushion 13N,Makeup,1,109000.00,2026-08-13,Bandung,Tokopedia,Completed,<NA>,109000.00
8,TKP-0111,MKO-MUP-001,Make Over Powerstay Weightless Liquid Foundati...,Makeup,1,169000.00,2026-08-24,Surabaya,Tokopedia,Completed,<NA>,169000.00
9,TKP-0112,WRD-MUP-001,Wardah Colorfit Velvet Matte Lip Mousse 03,Makeup,2,125800.00,2026-06-02,Surabaya,Tokopedia,Completed,<NA>,62900.00


Tersimpan: C:\Users\ADVAN\OneDrive - Universitas Teknologi Yogyakarta\Rinaldi\Ecommerce Sales\data\processed\clean
